In [6]:
import os
import sys
import subprocess

print("=" * 60)
print("SETUP")
print("=" * 60)

# Detect environment
on_colab = 'google.colab' in sys.modules
print(f"Environment: {'Colab' if on_colab else 'Local'}")

# Navigate to project root
os.chdir('/content/trading-signals' if on_colab else '.')
project_dir = os.getcwd()
print(f"Project dir: {project_dir}")

# Clone/pull repo if on Colab
if on_colab:
    if not os.path.exists('.git'):
        print("Cloning repo...")
        os.system('cd /content && rm -rf trading-signals && git clone https://github.com/tom-curtis/trading-signals.git')
    print("Pulling latest...")
    os.system('git pull origin main -q')
    data_path = '/root/.kaggle/datasets'
    os.makedirs(data_path, exist_ok=True)
else:
    data_path = 'data'
    os.makedirs(data_path, exist_ok=True)

# Download Kaggle data if missing
if not os.path.exists(f'{data_path}/Combined_News_DJIA.csv'):
    print(f"Downloading Kaggle data to {data_path}...")
    os.system(f'pip install kaggle -q')
    os.system(f'kaggle datasets download -d aaron7sun/stocknews -p {data_path} --unzip -q')

# Add project to path
sys.path.insert(0, project_dir)

# Validate setup
print(f"✓ Data: {os.path.exists(f'{data_path}/Combined_News_DJIA.csv')}")
print(f"✓ src.models: ", end="")
try:
    from src.data_loader import load_kaggle_data
    print("OK")
except:
    print("FAIL")

print("=" * 60)
print("Setup complete!\n")

SETUP
Environment: Colab
Project dir: /content/trading-signals
Pulling latest...
✓ Data: True
✓ src.models: OK
Setup complete!



In [7]:
from src.data_loader import load_kaggle_data, create_headline_examples, split_data, tokenize_and_pad, get_headline_stats

# Load data
news_df, prices_df = load_kaggle_data(f'{data_path}/Combined_News_DJIA.csv',
                                       f'{data_path}/upload_DJIA_table.csv')

# Create examples (keep headlines separate, ranked by importance)
headlines, labels = create_headline_examples(news_df)
get_headline_stats(headlines)

# Split into train/val/test (70/15/15)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(headlines, labels)

# Tokenize and pad
X_train_pad, X_val_pad, X_test_pad, tokenizer = tokenize_and_pad(X_train, X_val, X_test)

print(f"\nPhase 1 data ready!")
print(f"X_train: {X_train_pad.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val_pad.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test_pad.shape}, y_test: {y_test.shape}")

Loaded news: (1989, 27), prices: (1989, 7)
Created 49718 examples. Balance: 0.535 UP
Headlines: 49718, avg words: 17.7, max: 64
Train: 34802, Val: 7458, Test: 7458
Tokenized: vocab=37250, shapes=(34802, 500)

Phase 1 data ready!
X_train: (34802, 500), y_train: (34802,)
X_val: (7458, 500), y_val: (7458,)
X_test: (7458, 500), y_test: (7458,)


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

print("=" * 60)
print("BASELINES")
print("=" * 60)

# Baseline 1: Majority class
majority_class = np.bincount(y_train).argmax()
majority_pct = np.mean(y_test == majority_class)
print(f"\nBaseline 1: Majority Class")
print(f"  Always predict {majority_class}: {majority_pct:.4f}")

# Baseline 2: TF-IDF + Logistic Regression
print(f"\nBaseline 2: TF-IDF + LogisticRegression")
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, y_pred_lr)
print(f"  Test accuracy: {lr_acc:.4f}")

print("\n" + "=" * 60)

BASELINES

Baseline 1: Majority Class
  Always predict 1: 0.5354

Baseline 2: TF-IDF + LogisticRegression
  Test accuracy: 0.5156

